In [0]:
import pandas as pd
import numpy as np
import os, json, calendar
from datetime import datetime, date
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Widgets
# ═══════════════════════════════════════════════════════════════════════

_today = datetime.today()
_cq = f"{_today.year}-Q{(_today.month - 1) // 3 + 1}"
_y, _q = int(_cq[:4]), int(_cq[-1])
_total = (_y * 4 + _q - 1) + 1
_ny, _nq = divmod(_total, 4)
_default_tq = f"{_ny}-Q{_nq + 1}"

dbutils.widgets.dropdown(
    "target_quarter", _default_tq,
    [f"{y}-Q{q}" for y in range(datetime.today().year - 1, datetime.today().year + 1) for q in range(1, 5)],
    "Target Quarter",
)

dbutils.widgets.text("client_csv_filename", "proposed_csm_allocation.csv", "Client CSV Filename")
dbutils.widgets.text("take_rate_override_pct", "", "Take Rate Override (%)")

print("Widgets ready.")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Utilities — constants, helpers, formatters
# ═══════════════════════════════════════════════════════════════════════

QUARTER_MONTHS = {1: (1, 3), 2: (4, 6), 3: (7, 9), 4: (10, 12)}

PALETTE = [
    "#4A90E2", "#50A684", "#F5A623", "#D0021B",
    "#8E44AD", "#34495E", "#16A085", "#2C3E50",
    "#E67E22", "#27AE60", "#2980B9", "#C0392B",
]

BG_COLORS = {"SG": "#4A90E2", "HK": "#D0021B", "AU": "#50A684", "ID": "#F5A623"}
BU_COLORS = {"AspireBA": "#4A90E2", "AspireOS": "#50A684", "AspireX": "#F5A623"}
MOTION_COLORS = {"Key": "#8E44AD", "Scaled": "#16A085"}

def format_usd(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "$0.00"
    abs_val = abs(value)
    if abs_val >= 1_000_000_000:
        return f"${value / 1_000_000_000:,.2f}B"
    if abs_val >= 1_000_000:
        return f"${value / 1_000_000:,.2f}M"
    if abs_val >= 1_000:
        return f"${value / 1_000:,.2f}K"
    return f"${value:,.2f}"

def fmt_pct(v):
    return f"{v * 100:.2f}%" if v is not None and not (isinstance(v, float) and pd.isna(v)) else "0.00%"

def quarter_to_dates(q_str):
    year, q = int(q_str[:4]), int(q_str[-1])
    ms, me = QUARTER_MONTHS[q]
    return date(year, ms, 1), date(year, me, calendar.monthrange(year, me)[1])

def date_to_quarter(d):
    return f"{d.year}-Q{(d.month - 1) // 3 + 1}"

def current_quarter():
    return date_to_quarter(date.today())

def quarter_offset(q_str, offset):
    year, q = int(q_str[:4]), int(q_str[-1])
    total = (year * 4 + q - 1) + offset
    ny, nq = divmod(total, 4)
    return f"{ny}-Q{nq + 1}"

def q_to_int(q_str):
    y, q = int(q_str[:4]), int(q_str[-1])
    return y * 4 + q

def quarters_between(qs, qe):
    result, cur = [], qs
    while cur <= qe:
        result.append(cur)
        cur = quarter_offset(cur, 1)
    return result

TARGET_QUARTER  = dbutils.widgets.get("target_quarter")
CSV_FILENAME    = dbutils.widgets.get("client_csv_filename")
CURRENT_QUARTER = current_quarter()

# Resolve notebook folder path
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    NB_DIR = "/".join(_nb_path.rsplit("/", 1)[:-1])
except Exception:
    NB_DIR = "."

CSV_PATH = f"/Workspace{NB_DIR}/{CSV_FILENAME}"

print(f"Current Qtr : {CURRENT_QUARTER}")
print(f"Target Qtr  : {TARGET_QUARTER}")
print(f"CSV Path    : {CSV_PATH}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Load Client CSV (same folder as notebook)
# ═══════════════════════════════════════════════════════════════════════

EXPECTED_COLS = [
    "business_id", "business_ref_code", "company_name",
    "business_group", "business_unit", "account_motion", "account_manager",
]

client_df = None

# Pandas from Workspace path
try:
    client_pd = pd.read_csv(CSV_PATH)
    client_df = spark.createDataFrame(client_pd)
    print(f"Loaded via pd.read_csv: {CSV_PATH}")
except Exception as e1:
    print(f"Method 1 failed: {e1}")

if client_df is None:
    raise RuntimeError(f"Cannot load {CSV_FILENAME}. Ensure it's in the same repo folder as this notebook.")

# Normalise columns
for c in client_df.columns:
    client_df = client_df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
missing = [c for c in EXPECTED_COLS if c not in client_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}.  Found: {client_df.columns}")

client_df = client_df.select(EXPECTED_COLS)
clients_pd = client_df.toPandas()
clients_pd["business_id"] = clients_pd["business_id"].astype(str)

print(f"Clients loaded: {len(clients_pd)}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Revenue Data — global session cache
# ═══════════════════════════════════════════════════════════════════════

if "_REV_CACHE" not in dir():
    _client_ids = ",".join([f"'{x}'" for x in clients_pd["business_id"].unique()])
    # Use quarter_to_dates to get the end date as a string
    _, q_end = quarter_to_dates(CURRENT_QUARTER)
    q_end_str = q_end.strftime("%Y-%m-%d")

    _REV_CACHE = spark.sql(f"""
        SELECT
            CAST(r.business_id AS STRING) AS business_id,
            CAST(r.transaction_date AS DATE) AS txn_date,
            SUM(r.gross_revenue)     AS gross_revenue,
            SUM(r.gross_profit)      AS gross_profit
        FROM cat_legacy.analytics.revenue_core r
        WHERE r.category <> 'Credit'
          AND r.gross_revenue IS NOT NULL
          AND CAST(r.business_id AS STRING) IN ({_client_ids})
          AND r.transaction_date BETWEEN '2025-01-01' and '{q_end_str}'
        GROUP BY 1, 2
    """).toPandas()

    _REV_CACHE["txn_date"] = pd.to_datetime(_REV_CACHE["txn_date"])
    _REV_CACHE["year"]     = _REV_CACHE["txn_date"].dt.year
    _REV_CACHE["month"]    = _REV_CACHE["txn_date"].dt.month
    _REV_CACHE["quarter"]  = _REV_CACHE.apply(lambda r: f"{r['year']}-Q{(r['month']-1)//3+1}", axis=1)

    print(f"Revenue cache: {len(_REV_CACHE):,} rows")
else:
    print("Revenue cache already loaded — skipped re-fetch")

raw_rev = _REV_CACHE

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Quarterly Aggregations — Client level (enriched with client metadata)
# ═══════════════════════════════════════════════════════════════════════

# Merge revenue with client metadata
rev = raw_rev.merge(clients_pd, on="business_id", how="inner")

# Client-Quarter level
q_client = (
    rev.groupby(["business_id", "business_ref_code", "company_name",
                 "business_group", "business_unit", "account_motion",
                 "account_manager", "quarter"])
    .agg(gross_revenue=("gross_revenue", "sum"),
         gross_profit=("gross_profit", "sum"),
         active_days=("txn_date", "nunique"))
    .reset_index()
)
q_client["take_rate"] = np.where(
    q_client["gross_revenue"] != 0,
    q_client["gross_profit"] / q_client["gross_revenue"], 0
)

# Overall quarterly
q_overall = (
    rev.groupby("quarter")
    .agg(gross_revenue=("gross_revenue", "sum"),
         gross_profit=("gross_profit", "sum"))
    .reset_index()
    .sort_values("quarter")
)
q_overall["take_rate"] = np.where(
    q_overall["gross_revenue"] != 0,
    q_overall["gross_profit"] / q_overall["gross_revenue"], 0
)

print(f"Quarterly client rows : {len(q_client):,}")
print(f"Quarters available    : {sorted(q_client['quarter'].unique())}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Time Logic Engine — resolve L4Q, estimate current quarter if needed
# ═══════════════════════════════════════════════════════════════════════

def estimate_current_quarter_client(rev_df, clients_pd, current_q):
    """Estimate current (incomplete) quarter: actuals + daily run-rate projection."""
    today = date.today()
    q_start, q_end = quarter_to_dates(current_q)

    cq_data = rev_df[rev_df["quarter"] == current_q].copy()
    if cq_data.empty:
        return None

    current_month = today.month
    days_elapsed = today.day
    total_days_month = calendar.monthrange(today.year, current_month)[1]

    completed_months = [m for m in range(q_start.month, current_month)]
    future_months = [m for m in range(current_month + 1, q_end.month + 1)]

    # Completed months actuals
    completed = cq_data[cq_data["month"].isin(completed_months)].groupby("business_id").agg(
        rev_completed=("gross_revenue", "sum"), gp_completed=("gross_profit", "sum")
    ).reset_index() if completed_months else pd.DataFrame(columns=["business_id", "rev_completed", "gp_completed"])

    # Ongoing month run-rate
    ongoing = cq_data[cq_data["month"] == current_month].groupby("business_id").agg(
        rev_ongoing=("gross_revenue", "sum"), gp_ongoing=("gross_profit", "sum")
    ).reset_index()
    ongoing["rev_proj"] = ongoing["rev_ongoing"] / days_elapsed * total_days_month
    ongoing["gp_proj"]  = ongoing["gp_ongoing"]  / days_elapsed * total_days_month

    n_future = len(future_months)

    # Combine
    est = clients_pd[["business_id"]].copy()
    est = est.merge(completed, on="business_id", how="left").fillna(0)
    est = est.merge(ongoing[["business_id", "rev_proj", "gp_proj"]], on="business_id", how="left").fillna(0)
    est["gross_revenue"] = est["rev_completed"] + est["rev_proj"] + est["rev_proj"] * n_future
    est["gross_profit"]  = est["gp_completed"]  + est["gp_proj"]  + est["gp_proj"]  * n_future
    est["quarter"] = current_q
    est["take_rate"] = np.where(est["gross_revenue"] != 0, est["gross_profit"] / est["gross_revenue"], 0)
    return est[["business_id", "quarter", "gross_revenue", "gross_profit", "take_rate"]]


def resolve_l4q(target_q, current_q, q_client_df, rev_df, clients_pd_df):
    target_int  = q_to_int(target_q)
    current_int = q_to_int(current_q)
    available   = sorted(q_client_df["quarter"].unique())

    # CASE 1: past quarter
    if target_int <= current_int:
        candidates = [q for q in available if q_to_int(q) < target_int]
        l4q = candidates[-4:] if len(candidates) >= 4 else candidates
        return l4q, q_client_df

    # CASE 2: next immediate quarter
    if target_int == current_int + 1:
        est = estimate_current_quarter_client(rev_df, clients_pd_df, current_q)
        if est is not None:
            base = q_client_df[q_client_df["quarter"] != current_q].copy()
            est_full = est.merge(clients_pd_df, on="business_id", how="left")
            est_full["active_days"] = 0
            combined = pd.concat([base, est_full], ignore_index=True)
            past = [q for q in available if q_to_int(q) < current_int]
            l4q = (past[-3:] + [current_q]) if len(past) >= 3 else (past + [current_q])
        else:
            combined = q_client_df
            candidates = [q for q in available if q_to_int(q) < target_int]
            l4q = candidates[-4:]
        return l4q, combined

    # CASE 3: far future — iterative projection
    est = estimate_current_quarter_client(rev_df, clients_pd_df, current_q)
    if est is not None:
        base = q_client_df[q_client_df["quarter"] != current_q].copy()
        est_full = est.merge(clients_pd_df, on="business_id", how="left")
        est_full["active_days"] = 0
        combined = pd.concat([base, est_full], ignore_index=True)
    else:
        combined = q_client_df.copy()

    all_q = sorted(combined["quarter"].unique())
    last_known = all_q[-1] if all_q else quarter_offset(current_q, -1)
    intermediates = quarters_between(quarter_offset(last_known, 1), quarter_offset(target_q, -1))

    for proj_q in intermediates:
        avail_q = sorted(combined["quarter"].unique())
        proj_l4q = avail_q[-4:] if len(avail_q) >= 4 else avail_q
        l4q_sub = combined[combined["quarter"].isin(proj_l4q)]

        growth = _compute_client_growth(l4q_sub)
        last_q_data = combined[combined["quarter"] == avail_q[-1]].copy()
        last_q_data = last_q_data.merge(growth[["business_id", "avg_rev_g"]], on="business_id", how="left").fillna(0)
        last_q_data["gross_revenue"] = last_q_data["gross_revenue"] * (1 + last_q_data["avg_rev_g"])
        last_q_data["gross_profit"]  = last_q_data["gross_profit"]  * (1 + last_q_data["avg_rev_g"])
        last_q_data["quarter"] = proj_q
        last_q_data["take_rate"] = np.where(last_q_data["gross_revenue"] != 0, last_q_data["gross_profit"] / last_q_data["gross_revenue"], 0)
        last_q_data.drop(columns=["avg_rev_g"], inplace=True, errors="ignore")
        combined = pd.concat([combined, last_q_data], ignore_index=True)

    final_avail = sorted(combined["quarter"].unique())
    candidates = [q for q in final_avail if q_to_int(q) < target_int]
    l4q = candidates[-4:] if len(candidates) >= 4 else candidates
    return l4q, combined


print("Time Logic Engine ready")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# QoQ Growth & Target Revenue
# ═══════════════════════════════════════════════════════════════════════

def _compute_client_growth(q_df, min_rev_threshold=1000, growth_cap=5.0):
    """Compute simple average QoQ growth per client."""
    q_df = q_df.sort_values(["business_id", "quarter"])
    q_df["prev_rev"] = q_df.groupby("business_id")["gross_revenue"].shift(1)
    q_df["prev_gp"]  = q_df.groupby("business_id")["gross_profit"].shift(1)
    
    q_df["rev_g"] = np.where(
        (q_df["prev_rev"].notna()) & (q_df["prev_rev"] >= min_rev_threshold),
        (q_df["gross_revenue"] / q_df["prev_rev"] - 1),
        np.nan
    )
    q_df["gp_g"] = np.where(
        (q_df["prev_gp"].notna()) & (q_df["prev_gp"] >= min_rev_threshold),
        (q_df["gross_profit"] / q_df["prev_gp"] - 1),
        np.nan
    )
    
    growth = q_df.groupby("business_id").agg(
        avg_rev_g=("rev_g", "mean"),
        avg_gp_g =("gp_g",  "mean"),
        periods  =("rev_g", "count")
    ).reset_index()
    
    growth["avg_rev_g"] = growth["avg_rev_g"].clip(-0.99, growth_cap)
    growth["avg_gp_g"]  = growth["avg_gp_g"].clip(-0.99, growth_cap)
    
    return growth

# ── Execute ───────────────────────────────────────────────────────────
l4q_quarters, enriched_qc = resolve_l4q(TARGET_QUARTER, CURRENT_QUARTER, q_client, rev, clients_pd)
print(f"L4Q: {l4q_quarters}")

l4q_data = enriched_qc[enriched_qc["quarter"].isin(l4q_quarters)]

client_growth = _compute_client_growth(l4q_data)

last_q = sorted(l4q_quarters)[-1]
base_data = enriched_qc[enriched_qc["quarter"] == last_q].copy()

targets = base_data.merge(client_growth[["business_id", "avg_rev_g", "avg_gp_g", "periods"]], on="business_id", how="left").fillna(0)
targets["target_revenue"] = targets["gross_revenue"] * (1 + targets["avg_rev_g"])
targets["target_gp"]      = targets["gross_profit"]  * (1 + targets["avg_gp_g"])
targets["target_take_rate"] = np.where(targets["target_revenue"] != 0, targets["target_gp"] / targets["target_revenue"], 0)
targets["target_quarter"]   = TARGET_QUARTER
targets.rename(columns={"gross_revenue": "base_revenue", "gross_profit": "base_gp",
                         "take_rate": "base_take_rate", "quarter": "base_quarter"}, inplace=True)

# Historical avg take rate per client
hist_tr = l4q_data.groupby("business_id").agg(hist_avg_tr=("take_rate", "mean"), hist_avg_gp=("gross_profit", "mean")).reset_index()
targets = targets.merge(hist_tr, on="business_id", how="left")
overall_hist_tr = l4q_data.groupby("quarter").agg(rev=("gross_revenue", "sum"), gp=("gross_profit", "sum")).reset_index()
overall_hist_tr["tr"] = overall_hist_tr["gp"] / overall_hist_tr["rev"]
HIST_AVG_TR = overall_hist_tr["tr"].mean()
targets["hist_avg_tr"] = targets["hist_avg_tr"].fillna(HIST_AVG_TR)

# Guardrails
targets["guardrail"] = np.where(targets["target_take_rate"] >= targets["hist_avg_tr"], "Safe", "Risk")
targets["gp_trend"]  = np.where(targets["target_gp"] >= targets["hist_avg_gp"], "Growing", "Declining")
targets["tr_delta"]  = targets["target_take_rate"] - targets["hist_avg_tr"]

total_target_rev = targets["target_revenue"].sum()
total_target_gp  = targets["target_gp"].sum()
total_base_rev   = targets["base_revenue"].sum()
safe_count = (targets["guardrail"] == "Safe").sum()
risk_count = (targets["guardrail"] == "Risk").sum()
total_clients = len(targets)

print(f"Base Rev   : {format_usd(total_base_rev)}")
print(f"Target Rev : {format_usd(total_target_rev)}")
print(f"Target GP  : {format_usd(total_target_gp)}")
print(f"Safe/Risk  : {safe_count}/{risk_count}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Compute target allocations at every granularity
# ═══════════════════════════════════════════════════════════════════════

def build_allocation(l4q_df, group_cols, total_target):
    """Historical avg contribution → allocated target."""
    q_totals = l4q_df.groupby("quarter")["gross_revenue"].sum().reset_index().rename(columns={"gross_revenue": "q_total"})
    grp = l4q_df.groupby(group_cols + ["quarter"]).agg(
        rev=("gross_revenue", "sum"), gp=("gross_profit", "sum")
    ).reset_index()
    grp = grp.merge(q_totals, on="quarter")
    grp["contrib"] = grp["rev"] / grp["q_total"]
    avg = grp.groupby(group_cols).agg(
        avg_contrib=("contrib", "mean"),
        l4q_rev=("rev", "sum"), l4q_gp=("gp", "sum")
    ).reset_index()
    avg["target_rev"] = avg["avg_contrib"] * total_target
    avg["l4q_tr"] = np.where(avg["l4q_rev"] != 0, avg["l4q_gp"] / avg["l4q_rev"], 0)
    return avg.sort_values("target_rev", ascending=False).reset_index(drop=True)

alloc_bg     = build_allocation(l4q_data, ["business_group"], total_target_rev)
alloc_bu     = build_allocation(l4q_data, ["business_unit"], total_target_rev)
alloc_motion = build_allocation(l4q_data, ["account_motion"], total_target_rev)
alloc_csm    = build_allocation(l4q_data, ["account_manager"], total_target_rev)
alloc_bu_mot = build_allocation(l4q_data, ["business_unit", "account_motion"], total_target_rev)
alloc_csm_mot = build_allocation(l4q_data, ["account_manager", "account_motion"], total_target_rev)

# Client-level (targets df already has this)
alloc_client = targets[[
    "business_id", "business_ref_code", "company_name",
    "business_group", "business_unit", "account_motion", "account_manager",
    "base_quarter", "target_quarter",
    "base_revenue", "base_gp", "base_take_rate",
    "avg_rev_g", "avg_gp_g", "periods",
    "target_revenue", "target_gp", "target_take_rate",
    "hist_avg_tr", "tr_delta", "guardrail", "gp_trend"
]].sort_values("target_revenue", ascending=False).reset_index(drop=True)

print("Allocations computed for all granularities")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# What-If Simulation — Compute adjusted GP at every granularity
# ═══════════════════════════════════════════════════════════════════════

_tr_override_raw = dbutils.widgets.get("take_rate_override_pct").strip()
USE_OVERRIDE = len(_tr_override_raw) > 0
OVERRIDE_TR  = float(_tr_override_raw) / 100.0 if USE_OVERRIDE else None

if USE_OVERRIDE:
    sim = targets.copy()
    sim["sim_tr"] = OVERRIDE_TR
    sim["sim_gp"] = sim["target_revenue"] * OVERRIDE_TR
    sim["sim_guardrail"] = np.where(sim["sim_tr"] >= sim["hist_avg_tr"], "Safe", "Risk")
else:
    sim = targets.copy()
    sim["sim_tr"] = sim["target_take_rate"]
    sim["sim_gp"] = sim["target_gp"]
    sim["sim_guardrail"] = sim["guardrail"]

def sim_summary(df, group_cols=None):
    if group_cols:
        g = df.groupby(group_cols).agg(
            target_rev=("target_revenue", "sum"),
            sim_gp=("sim_gp", "sum"),
            clients=("business_id", "count"),
            safe=("sim_guardrail", lambda x: (x == "Safe").sum()),
            risk=("sim_guardrail", lambda x: (x == "Risk").sum()),
        ).reset_index()
    else:
        g = pd.DataFrame([{
            "target_rev": df["target_revenue"].sum(),
            "sim_gp": df["sim_gp"].sum(),
            "clients": len(df),
            "safe": (df["sim_guardrail"] == "Safe").sum(),
            "risk": (df["sim_guardrail"] == "Risk").sum(),
        }])
    g["sim_tr"] = np.where(g["target_rev"] != 0, g["sim_gp"] / g["target_rev"], 0)
    return g

sim_overall   = sim_summary(sim)
sim_bg        = sim_summary(sim, ["business_group"])
sim_bu        = sim_summary(sim, ["business_unit"])
sim_motion    = sim_summary(sim, ["account_motion"])
sim_csm       = sim_summary(sim, ["account_manager"])
sim_bu_mot    = sim_summary(sim, ["business_unit", "account_motion"])
sim_csm_mot   = sim_summary(sim, ["account_manager", "account_motion"])

print(f"Simulation: Override={'%.2f%%' % (OVERRIDE_TR*100) if USE_OVERRIDE else 'None'}")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# DASHBOARD — Hero Banner
# ═══════════════════════════════════════════════════════════════════════

displayHTML(f"""
<style>
  .hero-banner {{
    width: 100%; height: 100px; background: #ffffff;
    border: 0.5px solid #E0E0D8; border-radius: 16px; padding: 0 40px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.03);
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    display: flex; align-items: center; box-sizing: border-box;
  }}
  .hero-title-wrapper {{ display: flex; align-items: center; gap: 16px; white-space: nowrap; }}
  .hero-title-wrapper::before {{ content: ''; width: 35px; height: 3px; background: #000; border-radius: 2px; }}
  .hero-title {{ font-size: 22px; font-weight: 700; color: #000; text-transform: uppercase; letter-spacing: 0.15em; margin: 0; }}
</style>
<div class="hero-banner">
  <div class="hero-title-wrapper">
    <h1 class="hero-title">CSM Target Setting | {TARGET_QUARTER}</h1>
  </div>
</div>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Executive KPI Cards
# ═══════════════════════════════════════════════════════════════════════

growth_pct = ((total_target_rev / total_base_rev) - 1) * 100 if total_base_rev else 0
target_tr = total_target_gp / total_target_rev if total_target_rev else 0
overall_status = "Safe" if target_tr >= HIST_AVG_TR else "Risk"

kpis = json.dumps([
    {"label": "Total Clients",      "value": f"{total_clients:,}",           "sub": f"L4Q: {', '.join(l4q_quarters)}"},
    {"label": f"Base Revenue ({last_q})", "value": format_usd(total_base_rev), "sub": "Last completed quarter"},
    {"label": f"Target Revenue ({TARGET_QUARTER})", "value": format_usd(total_target_rev), "sub": f"Growth: {growth_pct:+.1f}%"},
    {"label": f"Target GP ({TARGET_QUARTER})",      "value": format_usd(total_target_gp),  "sub": f"Take Rate: {fmt_pct(target_tr)}"},
    {"label": "Hist Avg Take Rate",  "value": fmt_pct(HIST_AVG_TR),          "sub": f"L4Q average"},
    {"label": "Guardrail Status",    "value": f"{safe_count} Safe / {risk_count} Risk", "sub": overall_status},
])

displayHTML(f"""
<style>
  .kpi-row {{ display: flex; gap: 12px; width: 100%; }}
  .kpi-card {{
    flex: 1; background: #fff; border: 1px solid #F0F0EB; border-radius: 14px;
    padding: 22px 16px; box-shadow: 0 4px 12px rgba(0,0,0,0.03); text-align: center;
    font-family: 'Segoe UI', sans-serif;
  }}
  .kpi-label {{ font-size: 10px; font-weight: 700; color: #888780; text-transform: uppercase; letter-spacing: 0.1em; margin-bottom: 8px; }}
  .kpi-value {{ font-size: 24px; font-weight: 800; color: #1A1A18; line-height: 1; letter-spacing: -0.02em; }}
  .kpi-sub {{ font-size: 10px; color: #0C447C; font-weight: 500; margin-top: 6px; opacity: 0.8; }}
</style>
<div class="kpi-row" id="kpi-row"></div>
<script>
  (function() {{
    const data = {kpis};
    const container = document.getElementById('kpi-row');
    container.innerHTML = data.map(k => `
      <div class="kpi-card">
        <div class="kpi-label">${{k.label}}</div>
        <div class="kpi-value">${{k.value}}</div>
        <div class="kpi-sub">${{k.sub}}</div>
      </div>
    `).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# QoQ Revenue & GP Trend (native chart)
# ═══════════════════════════════════════════════════════════════════════

# Add target quarter to trend data
target_row = pd.DataFrame([{
    "quarter": TARGET_QUARTER,
    "gross_revenue": total_target_rev,
    "gross_profit": total_target_gp,
    "is_target": True,
}])
trend = q_overall.copy()
trend["is_target"] = False
trend = pd.concat([trend, target_row], ignore_index=True).sort_values("quarter")
trend["type"] = trend["is_target"].map({True: "Target", False: "Actual"})
trend["take_rate_pct"] = trend.apply(
    lambda r: round(r["gross_profit"] / r["gross_revenue"] * 100, 2) if r["gross_revenue"] else 0, axis=1
)

# Melt for stacked view
trend_melted = trend.melt(
    id_vars=["quarter", "type"],
    value_vars=["gross_revenue", "gross_profit"],
    var_name="metric", value_name="amount"
)
trend_melted["metric"] = trend_melted["metric"].map({"gross_revenue": "Revenue", "gross_profit": "Gross Profit"})

display(trend_melted)


In [0]:
displayHTML(f"""
<style>
  .viz-hero-banner {{
    height: 80px; background: #fff; border: 1px solid #E0E0D8; border-radius: 16px;
    padding: 0 30px; box-shadow: 0 4px 12px rgba(0,0,0,0.03);
    font-family: 'Segoe UI', sans-serif; display: flex; align-items: center;
  }}
  .hero-title-wrapper {{ display: flex; align-items: center; gap: 16px; }}
  .hero-title {{ font-size: 18px; font-weight: 800; color: #000; letter-spacing: 0.02em; margin: 0; }}
</style>
<div class="viz-hero-banner">
  <div class="hero-title-wrapper">
    <h1 class="hero-title">Target Allocations</h1>
  </div>
  <h3 style="font-size:12px;color:#5F5E5A;margin-left:8px;font-weight:600;">(All Views — {TARGET_QUARTER})</h3>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Target Allocation — By Business Group
# ═══════════════════════════════════════════════════════════════════════

_alloc_df = alloc_bg.copy()
_total = _alloc_df["target_rev"].sum()
_alloc_df["pct"] = np.where(_total > 0, _alloc_df["target_rev"] / _total * 100, 0)
_color_map = BG_COLORS

_items = []
for i, row in _alloc_df.iterrows():
    lbl = str(row["business_group"])
    _items.append({
        "label": lbl,
        "pct": round(row["pct"], 1),
        "amt": format_usd(row["target_rev"]),
        "l4q_tr": round(row.get("l4q_tr", 0), 4),
        "color": _color_map.get(lbl, PALETTE[i % len(PALETTE)]),
    })

_payload = json.dumps(_items)

displayHTML(f"""
<style>
  .alloc-card {{
    width: 780px; background: #fff; border: 1px solid #F0F0EB; border-radius: 16px;
    padding: 24px; box-shadow: 0 4px 20px rgba(0,0,0,0.04);
    font-family: 'Segoe UI', system-ui, sans-serif; box-sizing: border-box;
  }}
  .alloc-header {{ margin-bottom: 20px; border-bottom: 1.5px solid #F8F8F6; padding-bottom: 12px; }}
  .alloc-title {{ font-size: 17px; font-weight: 800; color: #1A1A18; margin: 0 0 4px; }}
  .alloc-subtitle {{ font-size: 10px; font-weight: 700; color: #A09F98; text-transform: uppercase; letter-spacing: 0.08em; }}
  .alloc-list {{ display: flex; flex-direction: column; }}
  .alloc-row {{ display: flex; align-items: center; padding: 10px 6px; border-radius: 8px; transition: background 0.2s; }}
  .alloc-row:hover {{ background: #F9F9F7; }}
  .alloc-label {{ flex: 0 0 180px; font-size: 13px; font-weight: 600; color: #444440; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }}
  .bar-container {{ flex: 1; padding: 0 12px; }}
  .bar-bg {{ width: 100%; height: 6px; background: #F1F1ED; border-radius: 10px; overflow: hidden; }}
  .bar-fill {{ height: 100%; border-radius: 10px; transition: width 0.6s ease; }}
  .alloc-val {{ flex: 0 0 200px; text-align: right; font-variant-numeric: tabular-nums; }}
  .pct-text {{ font-size: 13px; font-weight: 700; color: #1A1A18; }}
  .amt-text {{ color: #888780; font-weight: 500; font-size: 11px; margin-left: 6px; }}
</style>
<div class="alloc-card">
  <div class="alloc-header">
    <h3 class="alloc-title">By Business Group</h3>
    <span class="alloc-subtitle">% of {TARGET_QUARTER} target revenue</span>
  </div>
  <div id="alloc-list-business_group" class="alloc-list"></div>
</div>
<script>
  (function() {{
    const data = {_payload};
    const ct = document.getElementById('alloc-list-business_group');
    ct.innerHTML = data.map(item => {{
      let rowHtml = `
        <div class="alloc-row">
          <div class="alloc-label" title="${{item.label}}">${{item.label}}</div>
          <div class="bar-container">
            <div class="bar-bg"><div class="bar-fill" style="width:${{item.pct}}%;background:${{item.color}};"></div></div>
          </div>
          <div class="alloc-val">
            <span class="pct-text">${{item.pct}}%</span>
            <span class="amt-text">(${{item.amt}})</span>`;
      
        const trVal = item.l4q_tr !== undefined ? (item.l4q_tr * 100).toFixed(2) + '%' : '';
        rowHtml += `<span style="font-size:11px;color:#888780;margin-left:8px;">TR: ${{trVal}}</span>`;
        
      rowHtml += `</div></div>`;
      return rowHtml;
    }}).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Target Allocation — By Business Unit
# ═══════════════════════════════════════════════════════════════════════

_alloc_df = alloc_bu.copy()
_total = _alloc_df["target_rev"].sum()
_alloc_df["pct"] = np.where(_total > 0, _alloc_df["target_rev"] / _total * 100, 0)
_color_map = BU_COLORS

_items = []
for i, row in _alloc_df.iterrows():
    lbl = str(row["business_unit"])
    _items.append({
        "label": lbl,
        "pct": round(row["pct"], 1),
        "amt": format_usd(row["target_rev"]),
        "l4q_tr": round(row.get("l4q_tr", 0), 4),
        "color": _color_map.get(lbl, PALETTE[i % len(PALETTE)]),
    })

_payload = json.dumps(_items)

displayHTML(f"""
<style>
  .alloc-card {{
    width: 780px; background: #fff; border: 1px solid #F0F0EB; border-radius: 16px;
    padding: 24px; box-shadow: 0 4px 20px rgba(0,0,0,0.04);
    font-family: 'Segoe UI', system-ui, sans-serif; box-sizing: border-box;
  }}
  .alloc-header {{ margin-bottom: 20px; border-bottom: 1.5px solid #F8F8F6; padding-bottom: 12px; }}
  .alloc-title {{ font-size: 17px; font-weight: 800; color: #1A1A18; margin: 0 0 4px; }}
  .alloc-subtitle {{ font-size: 10px; font-weight: 700; color: #A09F98; text-transform: uppercase; letter-spacing: 0.08em; }}
  .alloc-list {{ display: flex; flex-direction: column; }}
  .alloc-row {{ display: flex; align-items: center; padding: 10px 6px; border-radius: 8px; transition: background 0.2s; }}
  .alloc-row:hover {{ background: #F9F9F7; }}
  .alloc-label {{ flex: 0 0 180px; font-size: 13px; font-weight: 600; color: #444440; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }}
  .bar-container {{ flex: 1; padding: 0 12px; }}
  .bar-bg {{ width: 100%; height: 6px; background: #F1F1ED; border-radius: 10px; overflow: hidden; }}
  .bar-fill {{ height: 100%; border-radius: 10px; transition: width 0.6s ease; }}
  .alloc-val {{ flex: 0 0 200px; text-align: right; font-variant-numeric: tabular-nums; }}
  .pct-text {{ font-size: 13px; font-weight: 700; color: #1A1A18; }}
  .amt-text {{ color: #888780; font-weight: 500; font-size: 11px; margin-left: 6px; }}
</style>
<div class="alloc-card">
  <div class="alloc-header">
    <h3 class="alloc-title">By Business Unit</h3>
    <span class="alloc-subtitle">% of {TARGET_QUARTER} target revenue</span>
  </div>
  <div id="alloc-list-business_unit" class="alloc-list"></div>
</div>
<script>
  (function() {{
    const data = {_payload};
    const ct = document.getElementById('alloc-list-business_unit');
    ct.innerHTML = data.map(item => {{
      let rowHtml = `
        <div class="alloc-row">
          <div class="alloc-label" title="${{item.label}}">${{item.label}}</div>
          <div class="bar-container">
            <div class="bar-bg"><div class="bar-fill" style="width:${{item.pct}}%;background:${{item.color}};"></div></div>
          </div>
          <div class="alloc-val">
            <span class="pct-text">${{item.pct}}%</span>
            <span class="amt-text">(${{item.amt}})</span>`;
      
        const trVal = item.l4q_tr !== undefined ? (item.l4q_tr * 100).toFixed(2) + '%' : '';
        rowHtml += `<span style="font-size:11px;color:#888780;margin-left:8px;">TR: ${{trVal}}</span>`;
        
      rowHtml += `</div></div>`;
      return rowHtml;
    }}).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Target Allocation — By Client Motion
# ═══════════════════════════════════════════════════════════════════════

_alloc_df = alloc_motion.copy()
_total = _alloc_df["target_rev"].sum()
_alloc_df["pct"] = np.where(_total > 0, _alloc_df["target_rev"] / _total * 100, 0)
_color_map = MOTION_COLORS

_items = []
for i, row in _alloc_df.iterrows():
    lbl = str(row["account_motion"])
    _items.append({
        "label": lbl,
        "pct": round(row["pct"], 1),
        "amt": format_usd(row["target_rev"]),
        "l4q_tr": round(row.get("l4q_tr", 0), 4),
        "color": _color_map.get(lbl, PALETTE[i % len(PALETTE)]),
    })

_payload = json.dumps(_items)

displayHTML(f"""
<style>
  .alloc-card {{
    width: 780px; background: #fff; border: 1px solid #F0F0EB; border-radius: 16px;
    padding: 24px; box-shadow: 0 4px 20px rgba(0,0,0,0.04);
    font-family: 'Segoe UI', system-ui, sans-serif; box-sizing: border-box;
  }}
  .alloc-header {{ margin-bottom: 20px; border-bottom: 1.5px solid #F8F8F6; padding-bottom: 12px; }}
  .alloc-title {{ font-size: 17px; font-weight: 800; color: #1A1A18; margin: 0 0 4px; }}
  .alloc-subtitle {{ font-size: 10px; font-weight: 700; color: #A09F98; text-transform: uppercase; letter-spacing: 0.08em; }}
  .alloc-list {{ display: flex; flex-direction: column; }}
  .alloc-row {{ display: flex; align-items: center; padding: 10px 6px; border-radius: 8px; transition: background 0.2s; }}
  .alloc-row:hover {{ background: #F9F9F7; }}
  .alloc-label {{ flex: 0 0 180px; font-size: 13px; font-weight: 600; color: #444440; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }}
  .bar-container {{ flex: 1; padding: 0 12px; }}
  .bar-bg {{ width: 100%; height: 6px; background: #F1F1ED; border-radius: 10px; overflow: hidden; }}
  .bar-fill {{ height: 100%; border-radius: 10px; transition: width 0.6s ease; }}
  .alloc-val {{ flex: 0 0 200px; text-align: right; font-variant-numeric: tabular-nums; }}
  .pct-text {{ font-size: 13px; font-weight: 700; color: #1A1A18; }}
  .amt-text {{ color: #888780; font-weight: 500; font-size: 11px; margin-left: 6px; }}
</style>
<div class="alloc-card">
  <div class="alloc-header">
    <h3 class="alloc-title">By Client Motion</h3>
    <span class="alloc-subtitle">% of {TARGET_QUARTER} target revenue</span>
  </div>
  <div id="alloc-list-account_motion" class="alloc-list"></div>
</div>
<script>
  (function() {{
    const data = {_payload};
    const ct = document.getElementById('alloc-list-account_motion');
    ct.innerHTML = data.map(item => {{
      let rowHtml = `
        <div class="alloc-row">
          <div class="alloc-label" title="${{item.label}}">${{item.label}}</div>
          <div class="bar-container">
            <div class="bar-bg"><div class="bar-fill" style="width:${{item.pct}}%;background:${{item.color}};"></div></div>
          </div>
          <div class="alloc-val">
            <span class="pct-text">${{item.pct}}%</span>
            <span class="amt-text">(${{item.amt}})</span>`;
      
        const trVal = item.l4q_tr !== undefined ? (item.l4q_tr * 100).toFixed(2) + '%' : '';
        rowHtml += `<span style="font-size:11px;color:#888780;margin-left:8px;">TR: ${{trVal}}</span>`;
        
      rowHtml += `</div></div>`;
      return rowHtml;
    }}).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Target Allocation — By CSM
# ═══════════════════════════════════════════════════════════════════════

_alloc_df = alloc_csm.head(20).copy()
_total = _alloc_df["target_rev"].sum()
_alloc_df["pct"] = np.where(_total > 0, _alloc_df["target_rev"] / _total * 100, 0)
_color_map = {}

_items = []
for i, row in _alloc_df.iterrows():
    lbl = str(row["account_manager"])
    _items.append({
        "label": lbl,
        "pct": round(row["pct"], 1),
        "amt": format_usd(row["target_rev"]),
        "l4q_tr": round(row.get("l4q_tr", 0), 4),
        "color": _color_map.get(lbl, PALETTE[i % len(PALETTE)]),
    })

_payload = json.dumps(_items)

displayHTML(f"""
<style>
  .alloc-card {{
    width: 100%; background: #fff; border: 1px solid #F0F0EB; border-radius: 16px;
    padding: 24px; box-shadow: 0 4px 20px rgba(0,0,0,0.04);
    font-family: 'Segoe UI', system-ui, sans-serif; box-sizing: border-box;
  }}
  .alloc-header {{ margin-bottom: 20px; border-bottom: 1.5px solid #F8F8F6; padding-bottom: 12px; }}
  .alloc-title {{ font-size: 17px; font-weight: 800; color: #1A1A18; margin: 0 0 4px; }}
  .alloc-subtitle {{ font-size: 10px; font-weight: 700; color: #A09F98; text-transform: uppercase; letter-spacing: 0.08em; }}
  .alloc-list {{ display: flex; flex-direction: column; }}
  .alloc-row {{ display: flex; align-items: center; padding: 10px 6px; border-radius: 8px; transition: background 0.2s; }}
  .alloc-row:hover {{ background: #F9F9F7; }}
  .alloc-label {{ flex: 0 0 180px; font-size: 13px; font-weight: 600; color: #444440; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }}
  .bar-container {{ flex: 1; padding: 0 12px; }}
  .bar-bg {{ width: 100%; height: 6px; background: #F1F1ED; border-radius: 10px; overflow: hidden; }}
  .bar-fill {{ height: 100%; border-radius: 10px; transition: width 0.6s ease; }}
  .alloc-val {{ flex: 0 0 200px; text-align: right; font-variant-numeric: tabular-nums; }}
  .pct-text {{ font-size: 13px; font-weight: 700; color: #1A1A18; }}
  .amt-text {{ color: #888780; font-weight: 500; font-size: 11px; margin-left: 6px; }}
</style>
<div class="alloc-card">
  <div class="alloc-header">
    <h3 class="alloc-title">By CSM</h3>
    <span class="alloc-subtitle">% of {TARGET_QUARTER} target revenue — Top managers</span>
  </div>
  <div id="alloc-list-account_manager" class="alloc-list"></div>
</div>
<script>
  (function() {{
    const data = {_payload};
    const ct = document.getElementById('alloc-list-account_manager');
    ct.innerHTML = data.map(item => {{
      let rowHtml = `
        <div class="alloc-row">
          <div class="alloc-label" title="${{item.label}}">${{item.label}}</div>
          <div class="bar-container">
            <div class="bar-bg"><div class="bar-fill" style="width:${{item.pct}}%;background:${{item.color}};"></div></div>
          </div>
          <div class="alloc-val">
            <span class="pct-text">${{item.pct}}%</span>
            <span class="amt-text">(${{item.amt}})</span>`;
      
        const trVal = item.l4q_tr !== undefined ? (item.l4q_tr * 100).toFixed(2) + '%' : '';
        rowHtml += `<span style="font-size:11px;color:#888780;margin-left:8px;">TR: ${{trVal}}</span>`;
        
      rowHtml += `</div></div>`;
      return rowHtml;
    }}).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Target Allocation — BU x Motion Cross-Tab
# ═══════════════════════════════════════════════════════════════════════

cross = alloc_bu_mot.copy()
cross["label"] = cross["business_unit"] + " / " + cross["account_motion"]
cross["pct"]   = np.where(total_target_rev > 0, cross["target_rev"] / total_target_rev * 100, 0)

items = []
for i, row in cross.iterrows():
    items.append({
        "label": row["label"],
        "pct": round(row["pct"], 1),
        "amt": format_usd(row["target_rev"]),
        "tr": fmt_pct(row["l4q_tr"]),
        "color": PALETTE[i % len(PALETTE)],
    })

payload = json.dumps(items)

displayHTML(f"""
<style>
  .xtab-card {{
    width: 100%; background: #fff; border: 1px solid #F0F0EB; border-radius: 16px;
    padding: 24px; box-shadow: 0 4px 20px rgba(0,0,0,0.04);
    font-family: 'Segoe UI', sans-serif;
  }}
  .xtab-header {{ margin-bottom: 20px; border-bottom: 1.5px solid #F8F8F6; padding-bottom: 12px; }}
  .xtab-title {{ font-size: 17px; font-weight: 800; color: #1A1A18; margin: 0 0 4px; }}
  .xtab-sub {{ font-size: 10px; font-weight: 700; color: #A09F98; text-transform: uppercase; letter-spacing: 0.08em; }}
  .xtab-grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(220px, 1fr)); gap: 12px; }}
  .xtab-item {{
    background: #FAFAF8; border: 1px solid #F0F0EB; border-radius: 12px; padding: 16px;
    text-align: center; transition: transform 0.2s;
  }}
  .xtab-item:hover {{ transform: translateY(-2px); box-shadow: 0 4px 10px rgba(0,0,0,0.05); }}
  .xtab-lbl {{ font-size: 11px; font-weight: 700; color: #888780; text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 6px; }}
  .xtab-val {{ font-size: 22px; font-weight: 800; color: #1A1A18; }}
  .xtab-pct {{ font-size: 12px; color: #0C447C; font-weight: 600; margin-top: 4px; }}
</style>
<div class="xtab-card">
  <div class="xtab-header">
    <h3 class="xtab-title">BU x Client Motion Cross-Tab</h3>
    <span class="xtab-sub">Target Revenue for {TARGET_QUARTER}</span>
  </div>
  <div id="xtab-grid" class="xtab-grid"></div>
</div>
<script>
  (function() {{
    const data = {payload};
    const ct = document.getElementById('xtab-grid');
    ct.innerHTML = data.map(item => `
      <div class="xtab-item">
        <div class="xtab-lbl">${{item.label}}</div>
        <div class="xtab-val">${{item.amt}}</div>
        <div class="xtab-pct">${{item.pct}}% of total  |  TR: ${{item.tr}}</div>
      </div>
    `).join('');
  }})();
</script>
""")


In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Target Allocation — CSM x Motion Table
# ═══════════════════════════════════════════════════════════════════════

csm_m = alloc_csm_mot.copy()
csm_m["target_rev_fmt"]  = csm_m["target_rev"].apply(format_usd)
csm_m["take_rate_fmt"]   = csm_m["l4q_tr"].apply(fmt_pct)
csm_m["contrib_pct"]     = (csm_m["avg_contrib"] * 100).round(1)

rows = []
for _, r in csm_m.sort_values("target_rev", ascending=False).iterrows():
    rows.append({
        "csm": r["account_manager"],
        "motion": r["account_motion"],
        "rev": r["target_rev_fmt"],
        "pct": f'{r["contrib_pct"]}%',
        "tr": r["take_rate_fmt"],
    })

payload = json.dumps(rows)

displayHTML(f"""
<style>
  .tbl-card {{
    width: 100%; background: #fff; border: 1px solid #F0F0EB; border-radius: 16px;
    padding: 24px; box-shadow: 0 4px 20px rgba(0,0,0,0.04);
    font-family: 'Segoe UI', sans-serif; max-height: 600px; overflow-y: auto;
  }}
  .tbl-header {{ margin-bottom: 16px; border-bottom: 1.5px solid #F8F8F6; padding-bottom: 12px; }}
  .tbl-title {{ font-size: 17px; font-weight: 800; color: #1A1A18; margin: 0 0 4px; }}
  .tbl-sub {{ font-size: 10px; font-weight: 700; color: #A09F98; text-transform: uppercase; }}
  table.csm-tbl {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
  table.csm-tbl th {{ text-align: left; font-size: 10px; font-weight: 700; color: #888780;
    text-transform: uppercase; letter-spacing: 0.08em; padding: 8px 12px; border-bottom: 2px solid #F0F0EB; }}
  table.csm-tbl td {{ padding: 10px 12px; border-bottom: 1px solid #F8F8F6; color: #1A1A18; }}
  table.csm-tbl tr:hover {{ background: #FAFAF8; }}
  .td-right {{ text-align: right; font-weight: 600; font-variant-numeric: tabular-nums; }}
</style>
<div class="tbl-card">
  <div class="tbl-header">
    <h3 class="tbl-title">CSM x Client Motion Detail</h3>
    <span class="tbl-sub">Target {TARGET_QUARTER} — sorted by revenue</span>
  </div>
  <table class="csm-tbl">
    <thead><tr><th>CSM</th><th>Motion</th><th style="text-align:right">Target Revenue</th><th style="text-align:right">Contribution</th><th style="text-align:right">L4Q Take Rate</th></tr></thead>
    <tbody id="csm-tbl-body"></tbody>
  </table>
</div>
<script>
  (function() {{
    const data = {payload};
    const tb = document.getElementById('csm-tbl-body');
    tb.innerHTML = data.map(r => `
      <tr>
        <td>${{r.csm}}</td>
        <td>${{r.motion}}</td>
        <td class="td-right">${{r.rev}}</td>
        <td class="td-right">${{r.pct}}</td>
        <td class="td-right">${{r.tr}}</td>
      </tr>
    `).join('');
  }})();
</script>
""")


In [0]:
displayHTML(f"""
<style>
  .viz-hero-banner {{
    height: 80px; background: #fff; border: 1px solid #E0E0D8; border-radius: 16px;
    padding: 0 30px; box-shadow: 0 4px 12px rgba(0,0,0,0.03);
    font-family: 'Segoe UI', sans-serif; display: flex; align-items: center;
  }}
  .hero-title-wrapper {{ display: flex; align-items: center; gap: 16px; }}
  .hero-title {{ font-size: 18px; font-weight: 800; color: #000; letter-spacing: 0.02em; margin: 0; }}
</style>
<div class="viz-hero-banner">
  <div class="hero-title-wrapper">
    <h1 class="hero-title">Profit Guardrails</h1>
  </div>
  <h3 style="font-size:12px;color:#5F5E5A;margin-left:8px;font-weight:600;">(Take Rate vs Historical Avg — {TARGET_QUARTER})</h3>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Guardrail Status Summary + Risk Clients Table
# ═══════════════════════════════════════════════════════════════════════

safe_pct = round(safe_count / total_clients * 100, 1) if total_clients else 0
risk_pct = round(risk_count / total_clients * 100, 1) if total_clients else 0

# Top risk clients
risk_df = alloc_client[alloc_client["guardrail"] == "Risk"].head(20)
risk_rows = []
for _, r in risk_df.iterrows():
    risk_rows.append({
        "name": r["company_name"],
        "bu": r["business_unit"],
        "csm": r["account_manager"],
        "rev": format_usd(r["target_revenue"]),
        "tgt_tr": fmt_pct(r["target_take_rate"]),
        "hist_tr": fmt_pct(r["hist_avg_tr"]),
        "delta": f'{r["tr_delta"]*100:+.2f}pp',
    })
risk_payload = json.dumps(risk_rows)

displayHTML(f"""
<style>
  .guard-wrapper {{ display: flex; gap: 16px; width: 100%; font-family: 'Segoe UI', sans-serif; }}
  .guard-card {{
    flex: 1; background: #fff; border: 1px solid #F0F0EB; border-radius: 16px;
    padding: 28px; box-shadow: 0 4px 12px rgba(0,0,0,0.03); text-align: center;
  }}
  .guard-label {{ font-size: 11px; font-weight: 700; color: #888780; text-transform: uppercase; letter-spacing: 0.12em; margin-bottom: 10px; }}
  .guard-val {{ font-size: 48px; font-weight: 800; line-height: 1; }}
  .guard-sub {{ font-size: 12px; font-weight: 600; margin-top: 8px; }}
  .safe {{ color: #166534; }}
  .risk {{ color: #991B1B; }}
  .risk-tbl-card {{
    width: 100%; background: #fff; border: 1px solid #F0F0EB; border-radius: 16px;
    padding: 24px; box-shadow: 0 4px 20px rgba(0,0,0,0.04); margin-top: 16px;
    font-family: 'Segoe UI', sans-serif; max-height: 500px; overflow-y: auto;
  }}
  .risk-tbl {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
  .risk-tbl th {{ text-align: left; font-size: 10px; font-weight: 700; color: #888780;
    text-transform: uppercase; padding: 8px 10px; border-bottom: 2px solid #F0F0EB; }}
  .risk-tbl td {{ padding: 8px 10px; border-bottom: 1px solid #F8F8F6; }}
  .risk-tbl tr:hover {{ background: #FEF2F2; }}
  .delta-neg {{ color: #991B1B; font-weight: 700; }}
</style>

<div class="guard-wrapper">
  <div class="guard-card">
    <div class="guard-label">Safe Clients</div>
    <div class="guard-val safe">{safe_count}</div>
    <div class="guard-sub safe">{safe_pct}% of portfolio</div>
  </div>
  <div class="guard-card">
    <div class="guard-label">At-Risk Clients</div>
    <div class="guard-val risk">{risk_count}</div>
    <div class="guard-sub risk">{risk_pct}% below hist. avg TR</div>
  </div>
  <div class="guard-card">
    <div class="guard-label">Hist Avg Take Rate</div>
    <div class="guard-val" style="color:#1A1A18;">{fmt_pct(HIST_AVG_TR)}</div>
    <div class="guard-sub" style="color:#0C447C;">L4Q benchmark</div>
  </div>
</div>

<div class="risk-tbl-card">
  <h3 style="font-size:17px;font-weight:800;color:#1A1A18;margin:0 0 12px;">Top Risk Clients — Below Historical Take Rate</h3>
  <table class="risk-tbl">
    <thead><tr><th>Company</th><th>BU</th><th>CSM</th><th style="text-align:right">Target Rev</th><th style="text-align:right">Target TR</th><th style="text-align:right">Hist TR</th><th style="text-align:right">Delta</th></tr></thead>
    <tbody id="risk-body"></tbody>
  </table>
</div>

<script>
  (function() {{
    const data = {risk_payload};
    document.getElementById('risk-body').innerHTML = data.map(r => `
      <tr>
        <td>${{r.name}}</td><td>${{r.bu}}</td><td>${{r.csm}}</td>
        <td class="td-right">${{r.rev}}</td><td class="td-right">${{r.tgt_tr}}</td>
        <td class="td-right">${{r.hist_tr}}</td><td class="td-right delta-neg">${{r.delta}}</td>
      </tr>
    `).join('');
  }})();
</script>
""")


In [0]:
displayHTML(f"""
<style>
  .viz-hero-banner {{
    height: 80px; background: #fff; border: 1px solid #E0E0D8; border-radius: 16px;
    padding: 0 30px; box-shadow: 0 4px 12px rgba(0,0,0,0.03);
    font-family: 'Segoe UI', sans-serif; display: flex; align-items: center;
  }}
  .hero-title-wrapper {{ display: flex; align-items: center; gap: 16px; }}
  .hero-title {{ font-size: 18px; font-weight: 800; color: #000; letter-spacing: 0.02em; margin: 0; }}
</style>
<div class="viz-hero-banner">
  <div class="hero-title-wrapper">
    <h1 class="hero-title">What-If Simulation</h1>
  </div>
  <h3 style="font-size:12px;color:#5F5E5A;margin-left:8px;font-weight:600;">(Adjust Take Rate Override widget above)</h3>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Simulation Results — All Granularities (Grid Layout)
# ═══════════════════════════════════════════════════════════════════════

def sim_to_cards(df, label_col=None):
    items = []
    for _, r in df.iterrows():
        lbl = str(r[label_col]) if label_col else "Overall"
        items.append({
            "label": lbl,
            "rev": format_usd(r["target_rev"]),
            "gp": format_usd(r["sim_gp"]),
            "tr": fmt_pct(r["sim_tr"]),
            "clients": int(r["clients"]),
            "safe": int(r["safe"]),
            "risk": int(r["risk"]),
        })
    return items

all_sims = {
    "Overall": sim_to_cards(sim_overall),
    "By Business Group": sim_to_cards(sim_bg, "business_group"),
    "By Business Unit": sim_to_cards(sim_bu, "business_unit"),
    "By Client Motion": sim_to_cards(sim_motion, "account_motion"),
    "By CSM (Top 15)": sim_to_cards(sim_csm.head(15), "account_manager"),
}

sim_payload = json.dumps(all_sims)
override_label = f"{OVERRIDE_TR*100:.2f}%" if USE_OVERRIDE else "None (using projected)"

displayHTML(f"""
<style>
  .sim-section {{ font-family: 'Segoe UI', sans-serif; }}
  .sim-override-badge {{
    display: inline-block; padding: 6px 16px; border-radius: 8px; font-size: 13px; font-weight: 700; margin-bottom: 16px;
    background: {'#FEF2F2' if USE_OVERRIDE else '#F7F7F5'};
    color: {'#991B1B' if USE_OVERRIDE else '#888780'};
    border: 1px solid {'#FCA5A5' if USE_OVERRIDE else '#E0E0D8'};
  }}
  .sim-group-title {{ font-size: 15px; font-weight: 800; color: #1A1A18; margin: 20px 0 10px; }}
  .sim-grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(240px, 1fr)); gap: 12px; margin-bottom: 16px; }}
  .sim-card {{
    background: #fff; border: 1px solid #F0F0EB; border-radius: 12px; padding: 16px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.02);
  }}
  .sim-card-label {{ font-size: 11px; font-weight: 700; color: #888780; text-transform: uppercase; letter-spacing: 0.05em; margin-bottom: 8px; }}
  .sim-card-rev {{ font-size: 20px; font-weight: 800; color: #1A1A18; }}
  .sim-card-gp {{ font-size: 16px; font-weight: 700; color: #50A684; margin-top: 4px; }}
  .sim-card-meta {{ font-size: 11px; color: #888780; margin-top: 6px; }}
  .sim-badge {{ display: inline-block; padding: 2px 6px; border-radius: 4px; font-size: 10px; font-weight: 700; }}
  .sim-badge.safe {{ background: #f0fdf4; color: #166534; }}
  .sim-badge.risk {{ background: #fef2f2; color: #991b1b; }}
</style>
<div class="sim-section">
  <div class="sim-override-badge">Take Rate Override: {override_label}</div>
  <div id="sim-container"></div>
</div>
<script>
  (function() {{
    const data = {sim_payload};
    const ct = document.getElementById('sim-container');
    let html = '';
    for (const [group, items] of Object.entries(data)) {{
      html += `<div class="sim-group-title">${{group}}</div><div class="sim-grid">`;
      items.forEach(item => {{
        html += `
          <div class="sim-card">
            <div class="sim-card-label">${{item.label}}</div>
            <div class="sim-card-rev">${{item.rev}}</div>
            <div class="sim-card-gp">GP: ${{item.gp}}</div>
            <div class="sim-card-meta">
              TR: ${{item.tr}} &nbsp;|&nbsp; ${{item.clients}} clients &nbsp;
              <span class="sim-badge safe">${{item.safe}} safe</span>
              <span class="sim-badge risk">${{item.risk}} risk</span>
            </div>
          </div>`;
      }});
      html += '</div>';
    }}
    ct.innerHTML = html;
  }})();
</script>
""")


In [0]:
displayHTML(f"""
<style>
  .viz-hero-banner {{
    height: 80px; background: #fff; border: 1px solid #E0E0D8; border-radius: 16px;
    padding: 0 30px; box-shadow: 0 4px 12px rgba(0,0,0,0.03);
    font-family: 'Segoe UI', sans-serif; display: flex; align-items: center;
  }}
  .hero-title-wrapper {{ display: flex; align-items: center; gap: 16px; }}
  .hero-title {{ font-size: 18px; font-weight: 800; color: #000; letter-spacing: 0.02em; margin: 0; }}
</style>
<div class="viz-hero-banner">
  <div class="hero-title-wrapper">
    <h1 class="hero-title">Client-Level Target Detail</h1>
  </div>
  <h3 style="font-size:12px;color:#5F5E5A;margin-left:8px;font-weight:600;">(Full exportable table)</h3>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Client-Level Target Table (native display for filtering/sorting)
# ═══════════════════════════════════════════════════════════════════════

export_df = alloc_client.copy()
export_df["target_revenue"] = export_df["target_revenue"].round(2)
export_df["target_gp"]      = export_df["target_gp"].round(2)
export_df["target_take_rate"] = (export_df["target_take_rate"] * 100).round(2)
export_df["hist_avg_tr"]    = (export_df["hist_avg_tr"] * 100).round(2)
export_df["avg_rev_g"]      = (export_df["avg_rev_g"] * 100).round(2)

display(spark.createDataFrame(export_df))


In [0]:
displayHTML(f"""
<style>
  .viz-hero-banner {{
    height: 80px; background: #fff; border: 1px solid #E0E0D8; border-radius: 16px;
    padding: 0 30px; box-shadow: 0 4px 12px rgba(0,0,0,0.03);
    font-family: 'Segoe UI', sans-serif; display: flex; align-items: center;
  }}
  .hero-title-wrapper {{ display: flex; align-items: center; gap: 16px; }}
  .hero-title {{ font-size: 18px; font-weight: 800; color: #000; letter-spacing: 0.02em; margin: 0; }}
</style>
<div class="viz-hero-banner">
  <div class="hero-title-wrapper">
    <h1 class="hero-title">Export</h1>
  </div>
  <h3 style="font-size:12px;color:#5F5E5A;margin-left:8px;font-weight:600;">(Download final target tables)</h3>
</div>
""")

In [0]:
# ═══════════════════════════════════════════════════════════════════════
# Export — save CSV to the same repo folder
# ═══════════════════════════════════════════════════════════════════════

output_name = f"csm_targets_{TARGET_QUARTER.replace('-','_')}.csv"
output_path = f"/Workspace{NB_DIR}/{output_name}"

try:
    alloc_client.to_csv(output_path, index=False)
    print(f"Exported: {output_path}")
except Exception as e:
    # Fallback: write to /tmp and tell user
    fallback = f"/tmp/{output_name}"
    alloc_client.to_csv(fallback, index=False)
    print(f"Exported to fallback: {fallback}")
    print(f"(Could not write to repo: {e})")

print(f"\nTotal rows: {len(alloc_client)}")
print(f"Columns: {list(alloc_client.columns)}")
